In [3]:
# ============================================
# Notebook: Revisión de estructura (Gold Parquet)
# Autor: (tu nombre)
# Objetivo:
# - Cargar archivos parquet
# - Revisar: dimensiones, columnas, dtypes, nulos, duplicados
# - Revisar: cardinalidad, rangos, outliers simples, columnas tipo fecha
# - Generar un reporte rápido por dataset
# ============================================

# --- Celda 1: Instalación opcional (si hace falta) ---
# !pip install -q pandas pyarrow fastparquet numpy

# --- Celda 2: Imports y configuración ---
import os
import re
import math
import json
import numpy as np
import pandas as pd
from datetime import datetime

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)

# --- Celda 3: Rutas de archivos (ya montados en este entorno) ---
FILES = [
    r"C:\Users\jagua\OneDrive\Documentos\Diplomado IA y TA\Modulo 4  Proyecto Integrador\ciber_monitoring\data_engineering\gold\gold_correlation_matrix_data.parquet",
    r"C:\Users\jagua\OneDrive\Documentos\Diplomado IA y TA\Modulo 4  Proyecto Integrador\ciber_monitoring\data_engineering\gold\gold_correlation_values.parquet",
    r"C:\Users\jagua\OneDrive\Documentos\Diplomado IA y TA\Modulo 4  Proyecto Integrador\ciber_monitoring\data_engineering\gold\gold_fact_risk_sentiment_daily.parquet",
    r"C:\Users\jagua\OneDrive\Documentos\Diplomado IA y TA\Modulo 4  Proyecto Integrador\ciber_monitoring\data_engineering\gold\gold_social_metrics_daily.parquet",
    r"C:\Users\jagua\OneDrive\Documentos\Diplomado IA y TA\Modulo 4  Proyecto Integrador\ciber_monitoring\data_engineering\gold\gold_storytelling_history.parquet",
    r"C:\Users\jagua\OneDrive\Documentos\Diplomado IA y TA\Modulo 4  Proyecto Integrador\ciber_monitoring\data_engineering\gold\gold_tech_edge_score.parquet",
    r"C:\Users\jagua\OneDrive\Documentos\Diplomado IA y TA\Modulo 4  Proyecto Integrador\ciber_monitoring\data_engineering\gold\gold_test_append.parquet",
    r"C:\Users\jagua\OneDrive\Documentos\Diplomado IA y TA\Modulo 4  Proyecto Integrador\ciber_monitoring\data_engineering\gold\gold_vulnerability_index.parquet",
]

# --- Celda 4: Helpers ---
def human_bytes(n: int) -> str:
    if n is None or pd.isna(n):
        return "NA"
    step = 1024.0
    units = ["B", "KB", "MB", "GB", "TB"]
    i = 0
    x = float(n)
    while x >= step and i < len(units) - 1:
        x /= step
        i += 1
    return f"{x:.2f} {units[i]}"

def safe_read_parquet(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"No existe el archivo: {path}")
    # engine auto; si falla, intenta pyarrow explícitamente
    try:
        return pd.read_parquet(path)
    except Exception as e1:
        try:
            return pd.read_parquet(path, engine="pyarrow")
        except Exception as e2:
            raise RuntimeError(f"Error leyendo {path}\n1) {e1}\n2) {e2}")

def infer_date_columns(df: pd.DataFrame):
    """
    Detecta columnas candidatas a fecha/hora:
    - dtype datetime
    - nombre contiene date/datetime/time/timestamp/fecha/hora
    - o strings que parecen ISO (heurística ligera)
    """
    candidates = []
    name_pat = re.compile(r"(date|datetime|time|timestamp|fecha|hora)", re.IGNORECASE)

    for col in df.columns:
        s = df[col]
        if pd.api.types.is_datetime64_any_dtype(s):
            candidates.append(col)
            continue

        if name_pat.search(str(col)):
            candidates.append(col)
            continue

        # Heurística de strings: muestra pequeña
        if pd.api.types.is_object_dtype(s) and len(s) > 0:
            sample = s.dropna().astype(str).head(25)
            if sample.empty:
                continue
            # Si un buen porcentaje parece fecha ISO / yyyy-mm-dd / yyyy/mm/dd
            iso_like = sample.str.match(r"^\d{4}[-/]\d{2}[-/]\d{2}").mean()
            if iso_like >= 0.6:
                candidates.append(col)

    # sin duplicados conservando orden
    seen = set()
    out = []
    for c in candidates:
        if c not in seen:
            out.append(c)
            seen.add(c)
    return out

def summarize_dataframe(df: pd.DataFrame, dataset_name: str, top_k: int = 10) -> dict:
    """
    Genera métricas/resumen para un DF. Devuelve dict para reporte.
    """
    nrows, ncols = df.shape
    mem_bytes = int(df.memory_usage(deep=True).sum())

    dtypes = df.dtypes.astype(str)
    nulls = df.isna().sum()
    null_pct = (nulls / max(nrows, 1) * 100).round(2)

    # Duplicados (filas)
    dup_rows = int(df.duplicated().sum()) if nrows else 0
    dup_pct = round((dup_rows / max(nrows, 1)) * 100, 2)

    # Cardinalidad (nunique)
    nunique = df.nunique(dropna=True)

    # Detectar columnas de fecha e intentar convertir si son object
    date_cols = infer_date_columns(df)
    conversions = {}
    for col in date_cols:
        if pd.api.types.is_object_dtype(df[col]):
            before_na = df[col].isna().mean()
            converted = pd.to_datetime(df[col], errors="coerce", utc=False)
            after_na = converted.isna().mean()
            # Si mejora o queda razonable, conservar conversión
            # (ej. si no empeora mucho y hay al menos algunos convertidos)
            if converted.notna().sum() >= max(5, int(0.05 * nrows)) and after_na <= min(0.95, before_na + 0.20):
                df[col] = converted
                conversions[col] = "object -> datetime64[ns] (coerce)"

    # Stats numéricas
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    num_desc = None
    if num_cols:
        num_desc = df[num_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
        num_desc["missing"] = df[num_cols].isna().sum()
        num_desc["missing_pct"] = (num_desc["missing"] / max(nrows, 1) * 100).round(2)
        num_desc = num_desc.sort_values("missing_pct", ascending=False)

    # Categóricas (object, category, bool) - top values
    cat_cols = [c for c in df.columns if (pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_categorical_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c]))]
    top_values = {}
    for c in cat_cols[: min(len(cat_cols), 30)]:  # limita para no saturar
        vc = df[c].astype("object").value_counts(dropna=False).head(top_k)
        top_values[c] = vc

    report = {
        "dataset": dataset_name,
        "shape": (nrows, ncols),
        "memory": {"bytes": mem_bytes, "human": human_bytes(mem_bytes)},
        "columns": list(df.columns),
        "dtypes": dtypes.to_dict(),
        "nulls": {
            "count": nulls.to_dict(),
            "pct": null_pct.to_dict(),
        },
        "duplicates_rows": {"count": dup_rows, "pct": dup_pct},
        "nunique": nunique.to_dict(),
        "date_columns_detected": date_cols,
        "date_conversions_applied": conversions,
        "numeric_columns": num_cols,
        "categorical_columns": cat_cols,
    }
    return report, num_desc, top_values, df

def print_quick_report(report: dict, show_cols: int = 25):
    print("=" * 110)
    print(f"DATASET: {report['dataset']}")
    print(f"Shape: {report['shape'][0]:,} filas x {report['shape'][1]:,} columnas | Memoria: {report['memory']['human']}")
    print(f"Duplicados (filas): {report['duplicates_rows']['count']:,} ({report['duplicates_rows']['pct']}%)")
    print("-" * 110)

    # columnas / dtypes (resumen compacto)
    cols = report["columns"]
    print(f"Columnas (primeras {min(show_cols, len(cols))}/{len(cols)}): {cols[:show_cols]}")
    print("-" * 110)

    # Top nulos
    null_pct = pd.Series(report["nulls"]["pct"]).sort_values(ascending=False)
    top_null = null_pct.head(10)
    if (top_null > 0).any():
        print("Top 10 columnas con más % nulos:")
        display(pd.DataFrame({"null_pct": top_null}).style.format({"null_pct": "{:.2f}%"}))
    else:
        print("Nulos: 0% en todas las columnas (según pandas).")

    # Fechas
    if report["date_columns_detected"]:
        print("-" * 110)
        print("Columnas detectadas como fecha/hora:", report["date_columns_detected"])
        if report["date_conversions_applied"]:
            print("Conversiones aplicadas:", report["date_conversions_applied"])

def display_basic_views(df: pd.DataFrame, n: int = 5):
    print("\nVista previa (head):")
    display(df.head(n))
    print("\nInfo (resumen):")
    display(pd.DataFrame({"dtype": df.dtypes.astype(str), "nunique": df.nunique(dropna=True), "nulls": df.isna().sum()}))

# --- Celda 5: Cargar y revisar todo en lote ---
datasets = {}      # nombre -> df
reports = {}       # nombre -> dict
numeric_desc = {}  # nombre -> describe DF
top_values = {}    # nombre -> dict de series

for path in FILES:
    name = os.path.basename(path)
    print(f"\nCargando: {name}")
    df = safe_read_parquet(path)

    report, num_desc, tops, df = summarize_dataframe(df, dataset_name=name)
    datasets[name] = df
    reports[name] = report
    numeric_desc[name] = num_desc
    top_values[name] = tops

    print_quick_report(report)

print("\n✅ Listo. DataFrames disponibles en el dict: datasets['archivo.parquet']")

# --- Celda 6: Tabla comparativa rápida entre datasets ---
summary_rows = []
for name, rep in reports.items():
    summary_rows.append({
        "dataset": name,
        "rows": rep["shape"][0],
        "cols": rep["shape"][1],
        "memory": rep["memory"]["human"],
        "dup_rows": rep["duplicates_rows"]["count"],
        "dup_pct": rep["duplicates_rows"]["pct"],
        "num_cols": len(rep["numeric_columns"]),
        "cat_cols": len(rep["categorical_columns"]),
        "date_cols_detected": len(rep["date_columns_detected"]),
    })

summary_df = pd.DataFrame(summary_rows).sort_values(["rows", "cols"], ascending=False)
display(summary_df)

# --- Celda 7: Inspección detallada por dataset (elige uno) ---
DATASET_TO_INSPECT = "gold_fact_risk_sentiment_daily.parquet"  # <-- cambia esto

df = datasets[DATASET_TO_INSPECT]
display_basic_views(df, n=10)

# --- Celda 8: Describe numérico del dataset elegido ---
if numeric_desc[DATASET_TO_INSPECT] is not None:
    display(numeric_desc[DATASET_TO_INSPECT])
else:
    print("No hay columnas numéricas detectadas en este dataset.")

# --- Celda 9: Top valores de columnas categóricas (dataset elegido) ---
tops = top_values[DATASET_TO_INSPECT]
if tops:
    # Muestra solo algunas para no saturar
    for i, (col, vc) in enumerate(tops.items()):
        print(f"\nTop valores: {col}")
        display(vc.to_frame("count"))
        if i >= 9:
            print("... (limitado a 10 columnas categóricas en la vista).")
            break
else:
    print("No hay columnas categóricas (object/category/bool) relevantes o están vacías.")

# --- Celda 10: Chequeos típicos de calidad (claves, duplicados por columnas) ---
# Define posibles keys (ajusta según tu modelo)
POSSIBLE_KEYS = [
    ["date"], ["fecha"], ["day"], ["timestamp"],
    ["source", "date"],
    ["entity", "date"],
    ["topic", "date"],
]

def check_key_duplicates(df: pd.DataFrame, keys: list[str]):
    keys = [k for k in keys if k in df.columns]
    if not keys:
        return None
    dup = df.duplicated(subset=keys).sum()
    return {"keys": keys, "dup_count": int(dup), "dup_pct": round(dup / max(len(df), 1) * 100, 3)}

key_checks = []
for keys in POSSIBLE_KEYS:
    res = check_key_duplicates(df, keys)
    if res:
        key_checks.append(res)

if key_checks:
    display(pd.DataFrame(key_checks).sort_values("dup_count", ascending=False))
else:
    print("No se encontraron combinaciones de keys candidatas presentes en el DF para evaluar duplicados por clave.")

# --- Celda 11: Exportar reporte JSON de estructura (opcional) ---
OUTPUT_JSON = r"C:\Users\jagua\OneDrive\Documentos\Diplomado IA y TA\Modulo 4  Proyecto Integrador\ciber_monitoring\tests\gold_structure_reports.json"

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(reports, f, ensure_ascii=False, indent=2, default=str)

print(f"📄 Reporte JSON guardado en: {OUTPUT_JSON}")



Cargando: gold_correlation_matrix_data.parquet
DATASET: gold_correlation_matrix_data.parquet
Shape: 1 filas x 3 columnas | Memoria: 207.00 B
Duplicados (filas): 0 (0.0%)
--------------------------------------------------------------------------------------------------------------
Columnas (primeras 3/3): ['date', 'days_risk_score', 'avg_sentiment']
--------------------------------------------------------------------------------------------------------------
Nulos: 0% en todas las columnas (según pandas).
--------------------------------------------------------------------------------------------------------------
Columnas detectadas como fecha/hora: ['date', 'avg_sentiment']

Cargando: gold_correlation_values.parquet
DATASET: gold_correlation_values.parquet
Shape: 28 filas x 6 columnas | Memoria: 5.06 KB
Duplicados (filas): 0 (0.0%)
--------------------------------------------------------------------------------------------------------------
Columnas (primeras 6/6): ['index', 'days_ri

C:\Users\jagua\AppData\Local\Temp\ipykernel_33840\1037613666.py:145: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  cat_cols = [c for c in df.columns if (pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_categorical_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c]))]
C:\Users\jagua\AppData\Local\Temp\ipykernel_33840\1037613666.py:145: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  cat_cols = [c for c in df.columns if (pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_categorical_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c]))]


,null_pct
days_risk_score,100.00%
avg_sentiment,7.14%
days_risk_score_total,7.14%
run_ts,7.14%
index,0.00%
run_id,0.00%


--------------------------------------------------------------------------------------------------------------
Columnas detectadas como fecha/hora: ['avg_sentiment', 'run_ts']

Cargando: gold_fact_risk_sentiment_daily.parquet
DATASET: gold_fact_risk_sentiment_daily.parquet
Shape: 235 filas x 6 columnas | Memoria: 28.82 KB
Duplicados (filas): 0 (0.0%)
--------------------------------------------------------------------------------------------------------------
Columnas (primeras 6/6): ['event_date', 'days_risk_score_total', 'avg_sentiment', 'asof_date', 'run_id', 'run_ts']
--------------------------------------------------------------------------------------------------------------
Nulos: 0% en todas las columnas (según pandas).
--------------------------------------------------------------------------------------------------------------
Columnas detectadas como fecha/hora: ['event_date', 'avg_sentiment', 'asof_date', 'run_ts']

Cargando: gold_social_metrics_daily.parquet
DATASET: gold_

C:\Users\jagua\AppData\Local\Temp\ipykernel_33840\1037613666.py:145: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  cat_cols = [c for c in df.columns if (pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_categorical_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c]))]
C:\Users\jagua\AppData\Local\Temp\ipykernel_33840\1037613666.py:145: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  cat_cols = [c for c in df.columns if (pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_categorical_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c]))]


...

,dtype,nunique,nulls
event_date,datetime64[ns],235,0
days_risk_score_total,float64,34,0
avg_sentiment,float64,7,0
asof_date,datetime64[ns],6,0
run_id,object,6,0
run_ts,datetime64[us],6,0


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max,missing,missing_pct
days_risk_score_total,235.0,784.893617,2849.985359,0.000,0.0,90.0,90.0,90.0,180.0,2781.0,19643.200000,19990.00,0,0.0
avg_sentiment,235.0,0.000491,0.070655,-0.775,0.0,0.0,0.0,0.0,0.0,0.0,0.037632,0.75,0,0.0



Top valores: run_id


,count
run_id,
29410ce1-84d3-45ae-9cbc-f9b292200af5,228
efb21744-65a1-4061-bdfb-f45c0c3be79f,3
d9530120-9cf1-49a2-9fe6-bf1065273173,1
d06f34f2-bd6d-445f-8c81-67dbc939a16b,1
a5263026-b0fe-4899-8e9b-eb510a6e7713,1
11a53e13-09f9-4373-93f2-91fa9517ee01,1


No se encontraron combinaciones de keys candidatas presentes en el DF para evaluar duplicados por clave.
📄 Reporte JSON guardado en: C:\Users\jagua\OneDrive\Documentos\Diplomado IA y TA\Modulo 4  Proyecto Integrador\ciber_monitoring\tests\gold_structure_reports.json


In [8]:
FILES[5]

'C:\\Users\\jagua\\OneDrive\\Documentos\\Diplomado IA y TA\\Modulo 4  Proyecto Integrador\\ciber_monitoring\\data_engineering\\gold\\gold_tech_edge_score.parquet'

In [4]:
name = FILES[5]
report, num_desc, tops, df = summarize_dataframe(df, dataset_name=name)

C:\Users\jagua\AppData\Local\Temp\ipykernel_33840\1037613666.py:145: DeprecationWarning: is_categorical_dtype is deprecated and will be removed in a future version. Use isinstance(dtype, pd.CategoricalDtype) instead
  cat_cols = [c for c in df.columns if (pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_categorical_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c]))]


In [6]:
df.head()

,event_date,days_risk_score_total,avg_sentiment,asof_date,run_id,run_ts
0,2024-01-01,100.0,-0.775,2026-01-15,efb21744-65a1-4061-bdfb-f45c0c3be79f,2026-01-15 22:50:51.180381
1,2024-01-02,200.0,0.000,2026-01-15,efb21744-65a1-4061-bdfb-f45c0c3be79f,2026-01-15 22:50:51.180381
2,2024-01-03,50.0,0.750,2026-01-15,efb21744-65a1-4061-bdfb-f45c0c3be79f,2026-01-15 22:50:51.180381
3,2026-01-20,17950.0,0.000,2026-01-20,d9530120-9cf1-49a2-9fe6-bf1065273173,2026-01-20 07:24:48.193339
4,2026-01-21,18970.0,0.000,2026-01-21,d06f34f2-bd6d-445f-8c81-67dbc939a16b,2026-01-21 05:24:53.464154


In [7]:
df.tail()

,event_date,days_risk_score_total,avg_sentiment,asof_date,run_id,run_ts
230,2026-01-23,19990.0,0.000000,2026-01-23,29410ce1-84d3-45ae-9cbc-f9b292200af5,2026-01-23 19:30:37.852778
231,2026-01-05,0.0,0.039601,2026-01-23,29410ce1-84d3-45ae-9cbc-f9b292200af5,2026-01-23 19:30:37.852778
232,2026-01-06,0.0,0.024583,2026-01-23,29410ce1-84d3-45ae-9cbc-f9b292200af5,2026-01-23 19:30:37.852778
233,2026-01-10,0.0,0.042291,2026-01-23,29410ce1-84d3-45ae-9cbc-f9b292200af5,2026-01-23 19:30:37.852778
234,2026-01-11,0.0,0.033810,2026-01-23,29410ce1-84d3-45ae-9cbc-f9b292200af5,2026-01-23 19:30:37.852778
